<a href="https://colab.research.google.com/github/DengDuangLang111/494-algo-representations/blob/week2/add_s11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def _():
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    from pathlib import Path

    # 1. Locate data directory
    current_dir = Path.cwd()
    possible_paths = [Path("../../data"), Path("data"), Path("D:/494/494-algo-representations/data")]
    DATA_DIR = None
    for p in possible_paths:
        if p.exists() and (p / "notes_labeled_final.parquet").exists():
            DATA_DIR = p
            break

    if DATA_DIR is None:
        raise FileNotFoundError("Data directory not found.")

    # 2. Read note status history
    history_file = DATA_DIR / "noteStatusHistory-00000.tsv"
    print(f"Reading history: {history_file.name} ...")

    df_history = pd.read_csv(
        history_file,
        sep="\t",
        usecols=["noteId", "currentStatus", "createdAtMillis"],
        dtype={"noteId": str, "currentStatus": str, "createdAtMillis": float},
        on_bad_lines='skip'
    )

    print("Extracting latest status...")
    df_latest = df_history.sort_values("createdAtMillis", ascending=False).drop_duplicates("noteId", keep="first")

    # Normalize status labels to match color mapping
    print("Normalizing status names...")
    df_latest["currentStatus"] = df_latest["currentStatus"].replace({
        "CURRENTLY_RATED_HELPFUL": "HELPFUL",
        "CURRENTLY_RATED_NOT_HELPFUL": "NOT_HELPFUL",
        "NEEDS_MORE_RATINGS": "NEEDS_MORE_RATINGS"
    })

    # 3. Load topics and merge
    df_topics = pd.read_parquet(DATA_DIR / "notes_labeled_final.parquet")
    df_topics = df_topics.explode("topic_labels").dropna(subset=["topic_labels"])
    df_topics["noteId"] = df_topics["noteId"].astype(str).str.strip()

    print("Merging data...")
    merged = df_topics.merge(df_latest[["noteId", "currentStatus"]], on="noteId", how="inner")
    print(f"   -> Merged {len(merged)} records")

    # 4. Plot outcomes
    if len(merged) > 0:
        print("Plotting...")
        counts = merged.groupby(["topic_labels", "currentStatus"]).size().unstack(fill_value=0)
        props = counts.div(counts.sum(axis=1), axis=0) * 100

        # Sort by Helpful percentage
        if "HELPFUL" in props.columns:
            props = props.sort_values("HELPFUL", ascending=False)
        else:
            props = props.sort_values(props.columns[0], ascending=False)

        props.index = props.index.str.replace("_", " ").str.title().str.replace(" & ", " And ")

        # Color mapping
        color_map = {
            "HELPFUL": "#4ade80",           # Green
            "NOT_HELPFUL": "#f87171",       # Red
            "NEEDS_MORE_RATINGS": "#60a5fa" # Blue
        }

        # Enforce column order
        col_order = ["HELPFUL", "NOT_HELPFUL", "NEEDS_MORE_RATINGS"]
        final_cols = [c for c in col_order if c in props.columns]

        fig, ax = plt.subplots(figsize=(16, 8))
        left_padding = pd.Series(0.0, index=props.index)

        for status in final_cols:
            data = props[status]
            c = color_map.get(status, "#999999")

            ax.barh(props.index, data, left=left_padding, color=c, label=status.replace("_", " "), height=0.8)

            # Annotate bars if percentage > 3%
            for idx, val in enumerate(data):
                if val > 3.0:
                    x_pos = left_padding.iloc[idx] + val / 2
                    # Use white text for red background, black otherwise
                    text_color = 'white' if status == "NOT_HELPFUL" else 'black'
                    ax.text(x_pos, idx, f"{val:.1f}%", va='center', ha='center', fontsize=9, color=text_color, fontweight='bold')

            left_padding += data

        ax.set_title(f"Community Note Outcomes by Topic (N={len(merged)})", fontsize=15)
        ax.set_xlabel("Outcomes (%)", fontsize=12)
        ax.set_xlim(0, 100)
        ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.05), ncol=3, frameon=False)

        sns.despine(left=True, bottom=False)
        ax.tick_params(axis='y', length=0)
        plt.subplots_adjust(left=0.25) # Adjust layout to prevent clipping

        save_path = DATA_DIR / "Figure_Outcome_Full.svg"
        plt.savefig(save_path, bbox_inches="tight")
        print(f"Chart saved to: {save_path}")
    return plt.show()

_()